# ModernBERT - Run the Pipeline from GitHub

One-shot entry point for the `Compartment` repo (`main.py` + `src/` package).

**How to use**

1. Edit the cell below: set `GITHUB_URL` to your repository (only needed if you are not using a GitHub input) and decide `USE_KAGGLE_INPUT`.
2. Make sure the **Compartment** dataset is attached as an input (it provides `dataset/` and `trial/`). 
3. Click **Run All**.

**What happens**

- locates/clones the repo (GitHub input mount or `git clone`)
- ensures the Python environment is ready
- runs `main.py <MODE>` through the CLI (config + validation happen there)
- prints metrics, previews the submission and creates `submission.zip`


## 1. Locate / clone the repository
Set the two variables at the top of this cell, then run it.


In [ ]:
import glob, json, os, subprocess, sys, zipfile

# ==== 1. Point at your repo ==========================================
USE_KAGGLE_INPUT = True   # True  -> GitHub repo mounted via "Add Input -> GitHub"
                          # False -> git clone the URL below into /kaggle/working
GITHUB_URL = 'https://github.com/AmnO-O/MoTune.git'   # <-- EDIT ME

# ==== 2. Locate the repo root ========================================
def sh(cmd, cwd=None):
    print('>>>', cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    sys.stdout.write(r.stdout)
    sys.stderr.write(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')

REPO = None
if USE_KAGGLE_INPUT:
    mains = sorted(glob.glob('/kaggle/input/github/**/main.py', recursive=True))
    if mains:
        REPO = os.path.dirname(mains[0])   # read-only snapshot; Kaggle input is fixed
if REPO is None:
    REPO = '/kaggle/working/compartment'
    if os.path.isdir(os.path.join(REPO, '.git')):
        # in-session clone from an earlier run -> refresh to the latest commit
        sh(f'git -C {REPO} fetch --depth 1 origin main')
        sh(f'git -C {REPO} reset --hard origin/main')
    elif not os.path.isdir(os.path.join(REPO, 'src')):
        sh(f'git clone --depth 1 {GITHUB_URL} {REPO}')

print('REPO =', REPO)
os.chdir(REPO)
sys.path.insert(0, REPO)
print('CWD =', os.getcwd())


## 2. Environment
Only installs on Kaggle if a dependency is missing (they all ship pre-installed).


In [ ]:
import subprocess, sys

# On Kaggle, torch / transformers are preinstalled. Only install when missing.
check = subprocess.run(
    [sys.executable, '-c', 'import torch, transformers, pandas, numpy, sklearn, scipy'],
    capture_output=True,
)
if check.returncode != 0:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'torch', 'transformers>=4.41', 'pandas', 'numpy', 'scikit-learn', 'scipy'],
        check=True,
    )
    print('Installed missing packages.')
else:
    print('Environment OK (torch, transformers, pandas, numpy, sklearn, scipy).')


## 3. Choose the run
Set `MODE` to `train80`, `train5` or `predict`. Append any CLI override to
`EXTRA_TRAIN` / `EXTRA_PREDICT` (e.g. `' --epochs 8 --batch 16 --ccc-weight 0.5'`).


In [ ]:
# ==== 3. The run ======================================================
MODE = 'train80'         # 'train80' = quick 80/20 split
                         # 'train5'  = stratified 5-fold CV (default)
                         # 'predict' = trial predictions + submission
DO_PREDICT = True        # run "main.py predict" after training (needs checkpoints)
EXTRA_TRAIN = ''         # optional CLI overrides, e.g. ' --epochs 8 --batch 16'
EXTRA_PREDICT = ''       # e.g. ' --predict-mode single --seed 7'
                         # predict-mode is auto-derived: 'single' after train80,
                         # '5fold' after train5 (only override if you know better)

# Data location is auto-detected (Kaggle dataset input or ./dataset) but can be
# pinned with ' --data-path <dir>'. Outputs go to /kaggle/working (or ./output).


## 3b. Config presets & overrides

Set `PROFILE` to `'reg'` (proven baseline recipe) or `'softmax'` (ordinal
bins → E[Y] with collapse-safe settings).

Both presets share the same encoder schedule (`unfreeze_from_layer=19`,
5 warm-up epochs + 7 fine-tuning epochs) and low learning rates so that
the encoder stays stable while the heads learn.

Add anything from the cheat-sheet into `OVERRIDES` to override a preset
value — your overrides win. `predict` uses the same file, so model
architecture knobs stay consistent automatically.

| key | reg | softmax | comment |
|---|---|---|---|
| head_mode | reg | softmax | |
| ccc_weight | 0.7 | 0.5 | blend of MSE and (1 − CCC) |
| lambda_rank | 0.5 | 0.5 | pairwise margin-ranking (boosts Spearman) |
| ce_weight | 0.0 | 0.3 | Gaussian soft-target CE (softmax only) |
| num_bins | 6 | 10 | finer softmax → smoother E[Y] |
| bin_sigma | 0.5 | 1.0 | Gaussian soft-target width |
| use_label_std | True | True | per-sample width from ModStd/HeadStd (softmax only) |
| amp_init_scale | 1024.0 | 1024.0 | GradScaler start; lower = fewer early fp16 overflows |
| amp_growth_interval | 256 | 256 | clean steps before scale recovers (default 2000 collapses) |
| num_epochs | 12 | 12 | |
| freeze_epochs | 4 | 4 | |
| unfreeze_from_layer | 18 | 18 | |


In [ ]:
# ==== 3b. Config presets ================================================
PROFILE = 'reg'           # 'reg'     -> proven regression recipe
                          # 'softmax' -> ordinal bins -> E[Y] (collapse-safe)

PRESETS = {
    'reg': {
        'head_mode': 'reg',
        'num_epochs': 12, 'freeze_epochs': 4, 'unfreeze_from_layer': 18,
        'batch_size': 32, 'head_lr': 5e-4, 'encoder_lr': 3e-6,
        'embedding_lr': 1e-5, 'ccc_weight': 0.7, 'lambda_rank': 0.5,
        'rank_margin': 0.5, 'ce_weight': 0.0, 'warmup_ratio': 0.15,
        'amp_init_scale': 1024.0, 'amp_growth_interval': 256, 'patience': 4,
    },
    'softmax': {
        'head_mode': 'softmax', 'num_bins': 10,   # wider bins, smoother E[Y]
        'ce_weight': 0.3,       # CE assists; MSE+CCC+rank carry the signal
        'bin_sigma': 1.0,       # wide Gaussian targets (no one-hot collapse)
        'num_epochs': 12, 'freeze_epochs': 4, 'unfreeze_from_layer': 18,
        'batch_size': 32, 'head_lr': 5e-4, 'encoder_lr': 3e-6,
        'embedding_lr': 1e-5, 'ccc_weight': 0.5, 'lambda_rank': 0.5,
        'rank_margin': 0.5, 'warmup_ratio': 0.15, 'amp_init_scale': 1024.0, 'amp_growth_interval': 256, 'patience': 4,
    },
}

# Optional overrides on top of the preset. Uncomment what you need.
OVERRIDES = {
    # 'batch_size': 16,          # effective batch = batch_size * accum_steps
    # 'accum_steps': 2,
    # 'num_epochs': 14,
    # 'freeze_epochs': 3,
    # 'unfreeze_from_layer': 18,
    # 'dropout': 0.2,
    # 'head_lr': 1e-4,           # lower LR for the heads
    # 'bin_sigma': 1.5,          # even softer targets (softmax only)
    # 'use_label_std': False,    # fixed bin_sigma instead of ModStd/HeadStd
    # 'seed': 42,
}

import json, os
from config import Config

merged = {**PRESETS.get(PROFILE, {}), **OVERRIDES}
CONFIG_PATH = os.path.join(os.getcwd(), 'config_run.json')

cfg = Config.defaults().update(**merged)
cfg.validate()
json.dump(merged, open(CONFIG_PATH, 'w'), indent=2)

print(f'Profile {PROFILE!r} + {len(OVERRIDES)} override(s) -> {CONFIG_PATH}')
print(json.dumps(merged, indent=2))

In [ ]:
import os, shlex, subprocess, sys

cfg_arg = ' --config ' + CONFIG_PATH if os.path.isfile(CONFIG_PATH) else ''

PY = sys.executable

if MODE not in ('train80', 'train5', 'predict'):
    raise SystemExit(f'Unknown MODE {MODE!r} (use train80 / train5 / predict)')

def run(cmd: str):
    print('>>>', cmd)
    subprocess.run(shlex.split(cmd), check=True, cwd=os.getcwd())

if MODE == 'predict':
    base = EXTRA_PREDICT or '--predict-mode single'
    run(f'{PY} main.py predict {base}{cfg_arg}')
else:
    run(f'{PY} main.py {MODE} {EXTRA_TRAIN}{cfg_arg}')
    if DO_PREDICT:
        # '5fold' needs fold{0..4}_best.pt (written by train5); train80 only
        # writes best.pt, so predict must use 'single' after train80.
        pm = EXTRA_PREDICT
        if 'predict-mode' not in pm:
            auto = 'single' if MODE == 'train80' else '5fold'
            pm = f'{pm.strip()} --predict-mode {auto}'.strip()
        print('\n--- now generating the submission ---')
        run(f'{PY} main.py predict {pm}{cfg_arg}')


## 4. Results
Printed from the JSON artifacts that `main.py` writes next to the checkpoints.


In [ ]:
import json, os

OUT = '/kaggle/working'
for fname in ('metrics.json', 'trial_metrics.json'):
    path = os.path.join(OUT, fname)
    if os.path.isfile(path):
        print('=' * 20, fname, '=' * 20)
        print(json.dumps(json.load(open(path)), indent=2))


## 5. Submission
Preview the predictions and get a Kaggle-ready `submission.zip` (TSV, no header).


In [ ]:
import os, zipfile

import pandas as pd

sub = os.path.join('/kaggle/working', 'submission', 'en-nn-trial-pred.tsv')
if not os.path.isfile(sub):
    print('No submission yet - run with MODE=predict or DO_PREDICT=True.')
else:
    df = pd.read_csv(sub, sep='\t', header=None, names=['tID', 'Modifier', 'Head'])
    print(df.head(10).to_string(index=False))
    print(f'\nTotal rows: {len(df)}')

    zip_path = os.path.join('/kaggle/working', 'submission', 'submission.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(sub, arcname=os.path.basename(sub))
    print('Kaggle-ready submission:', zip_path)
